# EgoRecall — 04: Visual Retrieval Baseline

**Problem 2:** Given a query image of an object (visual crop), retrieve the frame from the clip where that object was last seen.

**Models compared:**
| Model | Embedding Dim | Index Type |
|-------|--------------|------------|
| CLIP (ViT-B/32) | 512 | FAISS FlatIP (cosine) |
| BLIP ITM (base) | 768 | FAISS FlatIP (cosine) |

**Evaluation metric:** Top-1 frame retrieval accuracy
- Success = retrieved frame falls within ±5 seconds of the response track temporal window
- One FAISS index per clip — retrieval is clip-scoped

**Pipeline:**
```
Visual crop → Embed (CLIP/BLIP) → Query FAISS index → Top-1 frame → Compare to response track
```

**Key design decisions:**
- Index frames extracted at 1 FPS (every 30th video frame) on GCP VM
- Embeddings pre-computed and saved to GCS — load per clip during evaluation
- Peak memory = one clip's embeddings (~300 frames × 512/768 dims)


## 0 · Imports & Config

In [ ]:
!pip install faiss-gpu transformers torch torchvision \
    google-cloud-storage tqdm pandas pyarrow --quiet

In [ ]:
import json
import os
import io
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import faiss
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.notebook import tqdm
from google.cloud import storage
from transformers import (
    CLIPProcessor, CLIPModel,
    BlipProcessor, BlipForImageTextRetrieval
)

# ── GCS config ─────────────────────────────────────────────────────────────
BUCKET_NAME = "egorecall-data"

# ── Local paths ────────────────────────────────────────────────────────────
WORK_DIR    = Path("/content/egorecall_retrieval")
RESULTS_DIR = Path("/content/results/retrieval")
WORK_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Model config ───────────────────────────────────────────────────────────
CLIP_MODEL  = "openai/clip-vit-base-patch32"
BLIP_MODEL  = "Salesforce/blip-itm-base-coco"
CLIP_DIM    = 512
BLIP_DIM    = 768
BATCH_SIZE  = 64

# ── Evaluation config ──────────────────────────────────────────────────────
VIDEO_FPS        = 30       # source video FPS
INDEX_FPS        = 1        # index frame sampling rate
INDEX_STRIDE     = VIDEO_FPS // INDEX_FPS   # = 30
TOLERANCE_SEC    = 5        # ±5 seconds around response track = success
TOLERANCE_FRAMES = TOLERANCE_SEC * VIDEO_FPS  # = 150 video frames

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1 · Authentication & GCS

In [ ]:
from google.colab import auth
auth.authenticate_user()

gcs_client = storage.Client()
bucket     = gcs_client.bucket(BUCKET_NAME)
print(f"Connected to gs://{BUCKET_NAME}")

In [ ]:
# ── Load query-sets and raw annotations ───────────────────────────────────
bucket.blob("processed/vq_query_sets.parquet").download_to_filename("/tmp/vq_query_sets.parquet")
bucket.blob("ego4d/v2/annotations/vq_train.json").download_to_filename("/tmp/vq_train.json")
bucket.blob("ego4d/v2/annotations/vq_val.json").download_to_filename("/tmp/vq_val.json")

df = pd.read_parquet("/tmp/vq_query_sets.parquet")
with open("/tmp/vq_train.json") as f: train_raw = json.load(f)
with open("/tmp/vq_val.json")  as f: val_raw   = json.load(f)

print(f"Query-sets: {len(df):,}")
print(f"Clips     : {df['clip_uid'].nunique():,}")
print(f"Videos    : {df['video_uid'].nunique():,}")

In [ ]:
# ── Build response track lookup ────────────────────────────────────────────
# For evaluation: map (annotation_uid, qs_id) → response track video frame numbers
rt_lookup = {}  # (annotation_uid, qs_id) → [video_frame_numbers]

for split_raw in [train_raw, val_raw]:
    for video in split_raw["videos"]:
        for clip in video["clips"]:
            for anno in clip["annotations"]:
                for qs_id, qs in anno["query_sets"].items():
                    if not qs.get("is_valid"):
                        continue
                    key = (anno["annotation_uid"], qs_id)
                    rt_lookup[key] = [
                        box["video_frame_number"]
                        for box in qs.get("response_track", [])
                    ]

print(f"Response track entries: {len(rt_lookup):,}")

## 2 · Pre-compute Index Frame Embeddings (GCP VM)

**This section runs on the GCP VM, not Colab.**  
Copy the script below to the VM and run it in a tmux session overnight.

For each video:
1. Download video from GCS
2. Extract frames at 1 FPS (every 30th video frame)
3. Embed with CLIP and BLIP
4. Save per-clip embeddings + frame numbers to GCS
5. Delete local video

**Estimated time:** ~4-6 hours for all 1,727 videos on VM with GPU, ~12-15 hours CPU-only.

In [ ]:
# ── VM SCRIPT — copy this to the VM and run as: python3 embed_index_frames.py ──
VM_SCRIPT = '''
import json
import os
import shutil
import numpy as np
import torch
import cv2
from pathlib import Path
from tqdm import tqdm
from google.cloud import storage
from transformers import CLIPProcessor, CLIPModel, BlipProcessor, BlipForImageTextRetrieval
from PIL import Image

BUCKET_NAME  = "egorecall-data"
VIDEO_FPS    = 30
INDEX_STRIDE = 30    # 1 FPS
BATCH_SIZE   = 32
TMP_DIR      = Path("/tmp/embed_work")
TMP_DIR.mkdir(exist_ok=True)
CHECKPOINT   = Path("embed_checkpoint.json")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# Load models
print("Loading CLIP...")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model     = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE)
clip_model.eval()

print("Loading BLIP...")
blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-itm-base-coco")
blip_model     = BlipForImageTextRetrieval.from_pretrained("Salesforce/blip-itm-base-coco").to(DEVICE)
blip_model.eval()

gcs_client = storage.Client()
bucket     = gcs_client.bucket(BUCKET_NAME)

# Load workplan
with open("/tmp/vq_train.json") as f: train_raw = json.load(f)
with open("/tmp/vq_val.json")   as f: val_raw   = json.load(f)

# Build video → clips mapping
video_clips = {}  # video_uid → [(clip_uid, video_start_frame, video_end_frame)]
for split_raw in [train_raw, val_raw]:
    for video in split_raw["videos"]:
        vid = video["video_uid"]
        video_clips[vid] = [
            (c["clip_uid"], c["video_start_frame"], c["video_end_frame"])
            for c in video["clips"]
        ]

# Checkpoint
completed = set()
if CHECKPOINT.exists():
    with open(CHECKPOINT) as f:
        completed = set(json.load(f)["completed"])

def embed_clip(clip_model, clip_processor, blip_model, blip_processor,
               frames, frame_nos, batch_size=BATCH_SIZE):
    """Embed a list of PIL images in batches. Returns (clip_embs, blip_embs)."""
    clip_embs, blip_embs = [], []
    for i in range(0, len(frames), batch_size):
        batch = frames[i:i+batch_size]
        with torch.no_grad():
            # CLIP
            c_inputs = clip_processor(images=batch, return_tensors="pt",
                                      padding=True).to(DEVICE)
            c_feats  = clip_model.get_image_features(**c_inputs)
            c_feats  = c_feats / c_feats.norm(dim=-1, keepdim=True)
            clip_embs.append(c_feats.cpu().numpy())
            # BLIP
            b_inputs = blip_processor(images=batch, return_tensors="pt",
                                      padding=True).to(DEVICE)
            b_feats  = blip_model.vision_model(**b_inputs)["pooler_output"]
            b_feats  = b_feats / b_feats.norm(dim=-1, keepdim=True)
            blip_embs.append(b_feats.cpu().numpy())
    return np.vstack(clip_embs), np.vstack(blip_embs)


for video_uid, clips in tqdm(video_clips.items(), desc="Embedding videos"):
    if video_uid in completed:
        continue

    try:
        # Download video
        vid_path = TMP_DIR / f"{video_uid}.mp4"
        if not vid_path.exists():
            bucket.blob(f"ego4d/v2/video_540ss/{video_uid}.mp4").download_to_filename(
                str(vid_path)
            )

        # Collect all index frame numbers across all clips
        clip_frame_map = {}  # clip_uid → set of video_frame_nos
        for clip_uid, v_start, v_end in clips:
            clip_frame_map[clip_uid] = set(range(v_start, v_end, INDEX_STRIDE))

        all_needed = set()
        for fns in clip_frame_map.values():
            all_needed |= fns

        # Extract frames in one sequential pass
        cap = cv2.VideoCapture(str(vid_path))
        extracted = {}  # frame_no → PIL image
        max_fn = max(all_needed)
        fn = 0
        while fn <= max_fn:
            ret, frame = cap.read()
            if not ret:
                break
            if fn in all_needed:
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                extracted[fn] = Image.fromarray(rgb)
            fn += 1
        cap.release()

        # Per clip: embed and save
        for clip_uid, frame_nos in clip_frame_map.items():
            clip_frames = [(fn, extracted[fn]) for fn in sorted(frame_nos)
                           if fn in extracted]
            if not clip_frames:
                continue

            fns    = [f[0] for f in clip_frames]
            images = [f[1] for f in clip_frames]

            clip_embs, blip_embs = embed_clip(
                clip_model, clip_processor,
                blip_model, blip_processor,
                images, fns
            )
            fns_arr = np.array(fns, dtype=np.int32)

            # Save to GCS
            for model_name, embs in [("clip", clip_embs), ("blip", blip_embs)]:
                emb_buf = io.BytesIO()
                np.save(emb_buf, embs)
                emb_buf.seek(0)
                bucket.blob(f"embeddings/{model_name}/index/{clip_uid}.npy").upload_from_file(
                    emb_buf, content_type="application/octet-stream"
                )

            fns_buf = io.BytesIO()
            np.save(fns_buf, fns_arr)
            fns_buf.seek(0)
            bucket.blob(f"embeddings/clip/index/{clip_uid}_frames.npy").upload_from_file(
                fns_buf, content_type="application/octet-stream"
            )

        vid_path.unlink(missing_ok=True)
        completed.add(video_uid)
        if len(completed) % 50 == 0:
            with open(CHECKPOINT, "w") as f:
                json.dump({"completed": list(completed)}, f)

    except Exception as e:
        print(f"Error on {video_uid}: {e}")

with open(CHECKPOINT, "w") as f:
    json.dump({"completed": list(completed)}, f)
print(f"Done. {len(completed)} videos embedded.")
'''

# Save script to GCS for easy download on VM
blob = bucket.blob("processed/embed_index_frames.py")
blob.upload_from_string(VM_SCRIPT)
print("VM script saved to gs://egorecall-data/processed/embed_index_frames.py")
print("\nOn VM, run:")
print("  gsutil cp gs://egorecall-data/processed/embed_index_frames.py ~/")
print("  gsutil cp gs://egorecall-data/processed/vq_query_sets.parquet /tmp/")
print("  gsutil cp gs://egorecall-data/ego4d/v2/annotations/vq_train.json /tmp/")
print("  gsutil cp gs://egorecall-data/ego4d/v2/annotations/vq_val.json /tmp/")
print("  tmux new -s embed")
print("  python3 embed_index_frames.py")

## 3 · Pre-compute Visual Crop Embeddings (Colab)

Visual crops are the query images — one per query-set (18,111 total).  
These are small and fast to embed in Colab.

In [ ]:
# ── Load CLIP and BLIP ─────────────────────────────────────────────────────
print("Loading CLIP...")
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL)
clip_model     = CLIPModel.from_pretrained(CLIP_MODEL).to(DEVICE)
clip_model.eval()

print("Loading BLIP...")
blip_processor = BlipProcessor.from_pretrained(BLIP_MODEL)
blip_model     = BlipForImageTextRetrieval.from_pretrained(BLIP_MODEL).to(DEVICE)
blip_model.eval()

print("Models loaded.")

In [ ]:
# ── List all visual crops on GCS ───────────────────────────────────────────
vc_blobs = list(gcs_client.list_blobs(
    BUCKET_NAME,
    prefix="frames/retrieval/visual_crops/"
))
print(f"Visual crops on GCS: {len(vc_blobs):,}")
print(f"Sample: {vc_blobs[0].name}")

In [ ]:
# ── Parse annotation_uid and qs_id from filenames ─────────────────────────
# Filename format: {split}/{clip_uid}/{annotation_uid}_{qs_id}.jpg
vc_metadata = []
for blob in vc_blobs:
    parts = blob.name.split("/")
    # parts: ['frames', 'retrieval', 'visual_crops', split, clip_uid, filename]
    if len(parts) < 6:
        continue
    split    = parts[3]
    clip_uid = parts[4]
    filename = parts[5]  # annotation_uid_qs_id.jpg
    stem     = filename.replace(".jpg", "")
    # stem = annotation_uid_qs_id — qs_id is last character
    qs_id        = stem[-1]
    annotation_uid = stem[:-2]  # remove _N suffix
    vc_metadata.append({
        "blob_name"     : blob.name,
        "split"         : split,
        "clip_uid"      : clip_uid,
        "annotation_uid": annotation_uid,
        "qs_id"         : qs_id,
    })

vc_meta_df = pd.DataFrame(vc_metadata)
print(f"Parsed {len(vc_meta_df):,} visual crop entries")
print(vc_meta_df.head(3))

In [ ]:
# ── Embed all visual crops ─────────────────────────────────────────────────
def embed_images_batch(images, clip_model, clip_processor,
                        blip_model, blip_processor, batch_size=BATCH_SIZE):
    """Embed a list of PIL images. Returns (clip_embs, blip_embs) normalized."""
    clip_embs, blip_embs = [], []
    for i in range(0, len(images), batch_size):
        batch = images[i:i+batch_size]
        with torch.no_grad():
            # CLIP
            c_in   = clip_processor(images=batch, return_tensors="pt",
                                    padding=True).to(DEVICE)
            c_feat = clip_model.get_image_features(**c_in)
            c_feat = c_feat / c_feat.norm(dim=-1, keepdim=True)
            clip_embs.append(c_feat.cpu().numpy())
            # BLIP
            b_in   = blip_processor(images=batch, return_tensors="pt",
                                    padding=True).to(DEVICE)
            b_feat = blip_model.vision_model(**b_in)["pooler_output"]
            b_feat = b_feat / b_feat.norm(dim=-1, keepdim=True)
            blip_embs.append(b_feat.cpu().numpy())
    return np.vstack(clip_embs), np.vstack(blip_embs)


# Process in batches — download and embed
all_clip_embs = []
all_blip_embs = []
valid_meta    = []

LOAD_BATCH = 256  # download 256 images at a time
rows = vc_meta_df.to_dict("records")

for i in tqdm(range(0, len(rows), LOAD_BATCH), desc="Embedding visual crops"):
    batch_rows  = rows[i:i+LOAD_BATCH]
    batch_imgs  = []
    batch_valid = []

    for row in batch_rows:
        try:
            blob_data = bucket.blob(row["blob_name"]).download_as_bytes()
            img = Image.open(io.BytesIO(blob_data)).convert("RGB")
            batch_imgs.append(img)
            batch_valid.append(row)
        except Exception:
            continue

    if not batch_imgs:
        continue

    c_embs, b_embs = embed_images_batch(
        batch_imgs, clip_model, clip_processor,
        blip_model, blip_processor
    )
    all_clip_embs.append(c_embs)
    all_blip_embs.append(b_embs)
    valid_meta.extend(batch_valid)

clip_query_embs = np.vstack(all_clip_embs).astype(np.float32)
blip_query_embs = np.vstack(all_blip_embs).astype(np.float32)
query_meta_df   = pd.DataFrame(valid_meta)

print(f"\nEmbedded {len(query_meta_df):,} visual crops")
print(f"CLIP query embeddings : {clip_query_embs.shape}")
print(f"BLIP query embeddings : {blip_query_embs.shape}")

In [ ]:
# ── Save query embeddings to GCS ───────────────────────────────────────────
def save_npy_to_gcs(arr, gcs_path):
    buf = io.BytesIO()
    np.save(buf, arr)
    buf.seek(0)
    bucket.blob(gcs_path).upload_from_file(
        buf, content_type="application/octet-stream"
    )

save_npy_to_gcs(clip_query_embs, "embeddings/clip/queries.npy")
save_npy_to_gcs(blip_query_embs, "embeddings/blip/queries.npy")
query_meta_df.to_parquet("/tmp/query_metadata.parquet", index=False)
bucket.blob("embeddings/queries_metadata.parquet").upload_from_filename(
    "/tmp/query_metadata.parquet"
)

print("Saved to GCS:")
print(f"  gs://{BUCKET_NAME}/embeddings/clip/queries.npy")
print(f"  gs://{BUCKET_NAME}/embeddings/blip/queries.npy")
print(f"  gs://{BUCKET_NAME}/embeddings/queries_metadata.parquet")

## 4 · CLIP + FAISS Retrieval Evaluation

**Run this after the VM embedding job completes.**

For each query-set:
1. Load the clip's CLIP index embeddings from GCS
2. Build a FAISS FlatIP index (cosine similarity via normalized vectors)
3. Query with the visual crop embedding
4. Check if top-1 retrieved frame falls within ±5s of the response track

In [ ]:
# ── Load query embeddings and metadata ────────────────────────────────────
clip_query_embs = np.load(io.BytesIO(
    bucket.blob("embeddings/clip/queries.npy").download_as_bytes()
)).astype(np.float32)

query_meta_df = pd.read_parquet(io.BytesIO(
    bucket.blob("embeddings/queries_metadata.parquet").download_as_bytes()
))

print(f"Query embeddings : {clip_query_embs.shape}")
print(f"Query metadata   : {len(query_meta_df):,} rows")

In [ ]:
def load_clip_index(clip_uid, model_name="clip"):
    """
    Load pre-computed embeddings and frame numbers for one clip.
    Returns (embeddings: np.float32, frame_nos: np.int32) or (None, None).
    """
    try:
        emb_bytes = bucket.blob(
            f"embeddings/{model_name}/index/{clip_uid}.npy"
        ).download_as_bytes()
        fns_bytes = bucket.blob(
            f"embeddings/clip/index/{clip_uid}_frames.npy"
        ).download_as_bytes()
        embs = np.load(io.BytesIO(emb_bytes)).astype(np.float32)
        fns  = np.load(io.BytesIO(fns_bytes)).astype(np.int32)
        return embs, fns
    except Exception:
        return None, None


def build_faiss_index(embeddings):
    """Build a FAISS FlatIP index (cosine similarity on normalized vectors)."""
    dim   = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    if DEVICE == "cuda":
        res   = faiss.StandardGpuResources()
        index = faiss.index_cpu_to_gpu(res, 0, index)
    index.add(embeddings)
    return index


def is_success(retrieved_frame_no, rt_frame_nos, tolerance=TOLERANCE_FRAMES):
    """
    Success if retrieved frame is within tolerance frames of the response track window.
    rt_frame_nos: list of video_frame_numbers from the response track.
    """
    if not rt_frame_nos:
        return False
    rt_min = min(rt_frame_nos) - tolerance
    rt_max = max(rt_frame_nos) + tolerance
    return rt_min <= retrieved_frame_no <= rt_max


print("Evaluation functions defined.")

In [ ]:
# ── Run CLIP retrieval evaluation ─────────────────────────────────────────
def run_retrieval_eval(query_embs, query_meta_df, rt_lookup,
                        model_name="clip", split="val"):
    """
    Evaluate retrieval for one model across all query-sets in a split.
    Processes clip by clip to manage memory.
    """
    # Filter to split
    split_meta = query_meta_df[query_meta_df.split == split].reset_index(drop=True)
    split_embs = query_embs[split_meta.index]

    results = []
    clip_groups = split_meta.groupby("clip_uid")

    for clip_uid, group in tqdm(clip_groups, desc=f"{model_name} retrieval ({split})"):
        # Load index for this clip
        index_embs, index_fns = load_clip_index(clip_uid, model_name)
        if index_embs is None or len(index_embs) == 0:
            continue

        # Build FAISS index
        faiss_index = build_faiss_index(index_embs)

        # Query each visual crop in this clip
        for _, row in group.iterrows():
            query_idx = row.name
            q_emb     = query_embs[query_idx:query_idx+1]  # (1, dim)

            # Top-1 retrieval
            scores, indices = faiss_index.search(q_emb, k=1)
            top1_idx        = indices[0][0]
            top1_frame      = int(index_fns[top1_idx])
            top1_score      = float(scores[0][0])

            # Check success
            key     = (row["annotation_uid"], row["qs_id"])
            rt_fns  = rt_lookup.get(key, [])
            success = is_success(top1_frame, rt_fns)

            results.append({
                "clip_uid"       : clip_uid,
                "annotation_uid" : row["annotation_uid"],
                "qs_id"          : row["qs_id"],
                "retrieved_frame": top1_frame,
                "similarity"     : top1_score,
                "rt_min_frame"   : min(rt_fns) if rt_fns else None,
                "rt_max_frame"   : max(rt_fns) if rt_fns else None,
                "success"        : success,
                "model"          : model_name,
                "split"          : split,
            })

    results_df = pd.DataFrame(results)
    accuracy   = results_df["success"].mean()
    return results_df, accuracy


print("Running CLIP retrieval evaluation on val split...")
clip_results_df, clip_accuracy = run_retrieval_eval(
    clip_query_embs, query_meta_df, rt_lookup,
    model_name="clip", split="val"
)

print(f"\n── CLIP Retrieval Results (val) ─────────────────────")
print(f"  Queries evaluated : {len(clip_results_df):,}")
print(f"  Top-1 accuracy    : {clip_accuracy:.4f}")
print(f"  (±{TOLERANCE_SEC}s tolerance around response track)")
print(f"────────────────────────────────────────────────────")

## 5 · BLIP + FAISS Retrieval Evaluation

In [ ]:
# ── Load BLIP query embeddings ─────────────────────────────────────────────
blip_query_embs = np.load(io.BytesIO(
    bucket.blob("embeddings/blip/queries.npy").download_as_bytes()
)).astype(np.float32)

print(f"BLIP query embeddings: {blip_query_embs.shape}")

In [ ]:
print("Running BLIP retrieval evaluation on val split...")
blip_results_df, blip_accuracy = run_retrieval_eval(
    blip_query_embs, query_meta_df, rt_lookup,
    model_name="blip", split="val"
)

print(f"\n── BLIP Retrieval Results (val) ─────────────────────")
print(f"  Queries evaluated : {len(blip_results_df):,}")
print(f"  Top-1 accuracy    : {blip_accuracy:.4f}")
print(f"  (±{TOLERANCE_SEC}s tolerance around response track)")
print(f"────────────────────────────────────────────────────")

## 6 · Results Comparison

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────
summary = pd.DataFrame([
    {"Model": "CLIP (ViT-B/32) + FAISS", "Top-1 Accuracy": clip_accuracy},
    {"Model": "BLIP ITM + FAISS",         "Top-1 Accuracy": blip_accuracy},
])
summary["Top-1 Accuracy"] = summary["Top-1 Accuracy"].round(4)
print(summary.to_string(index=False))

In [ ]:
# ── Bar chart ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Accuracy comparison
models = ["CLIP\n+FAISS", "BLIP\n+FAISS"]
accs   = [clip_accuracy, blip_accuracy]
colors = ["#4C72B0", "#DD8452"]
bars   = axes[0].bar(models, accs, color=colors, edgecolor="white", width=0.5)
for bar, val in zip(bars, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f"{val:.3f}", ha="center", va="bottom", fontweight="bold")
axes[0].set_ylabel(f"Top-1 Accuracy (±{TOLERANCE_SEC}s)")
axes[0].set_title("Retrieval Accuracy", fontweight="bold")
axes[0].set_ylim(0, max(accs) * 1.3)

# Similarity score distribution
axes[1].hist(clip_results_df["similarity"], bins=50, alpha=0.7,
             color="#4C72B0", label="CLIP", edgecolor="white")
axes[1].hist(blip_results_df["similarity"], bins=50, alpha=0.7,
             color="#DD8452", label="BLIP", edgecolor="white")
axes[1].set_xlabel("Cosine similarity (query vs retrieved frame)")
axes[1].set_ylabel("Query-sets")
axes[1].set_title("Similarity Score Distribution", fontweight="bold")
axes[1].legend()

plt.suptitle("EgoRecall — Retrieval Baseline Results", fontweight="bold", fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "retrieval_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Breakdown by temporal gap quartile ────────────────────────────────────
# Does retrieval accuracy drop as the gap between query frame and response track grows?
combined = pd.concat([clip_results_df, blip_results_df])
combined["rt_center"] = (combined["rt_min_frame"] + combined["rt_max_frame"]) / 2

# Merge with query-set metadata to get query_video_frame
combined = combined.merge(
    df[["annotation_uid", "qs_id", "query_video_frame"]],
    on=["annotation_uid", "qs_id"],
    how="left"
)
combined["temporal_gap_sec"] = (
    combined["query_video_frame"] - combined["rt_center"]
).abs() / VIDEO_FPS

combined["gap_quartile"] = pd.qcut(
    combined["temporal_gap_sec"].clip(upper=200),
    q=4, labels=["Q1 (closest)", "Q2", "Q3", "Q4 (farthest)"]
)

gap_acc = combined.groupby(["model", "gap_quartile"])["success"].mean().unstack()
print("\nAccuracy by temporal gap quartile:")
print(gap_acc.round(3).to_string())

gap_acc.T.plot(kind="bar", figsize=(10, 4), color=colors)
plt.title("Retrieval Accuracy by Temporal Gap Quartile", fontweight="bold")
plt.xlabel("Temporal gap quartile")
plt.ylabel("Top-1 Accuracy")
plt.xticks(rotation=30)
plt.legend(title="Model")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "retrieval_by_gap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Save all results to GCS ───────────────────────────────────────────────
all_results = pd.concat([clip_results_df, blip_results_df])
all_results.to_parquet("/tmp/retrieval_results.parquet", index=False)
bucket.blob("processed/retrieval_results.parquet").upload_from_filename(
    "/tmp/retrieval_results.parquet"
)

summary.to_parquet("/tmp/retrieval_summary.parquet", index=False)
bucket.blob("processed/retrieval_summary.parquet").upload_from_filename(
    "/tmp/retrieval_summary.parquet"
)

print("Results saved to GCS:")
print(f"  gs://{BUCKET_NAME}/processed/retrieval_results.parquet")
print(f"  gs://{BUCKET_NAME}/processed/retrieval_summary.parquet")

---
### Next: `05_end_to_end_eval.ipynb`
Inputs from this notebook:
- `gs://egorecall-data/processed/retrieval_results.parquet`
- `gs://egorecall-data/embeddings/clip/` and `embeddings/blip/`
- `gs://egorecall-data/models/yolov8s_finetuned/best.pt` ← from notebook 03